## Abbreviation/Acronym Expansion in English Documents

### Goal

This notebook demonstrates how to normalize acronyms and abbreviations in English documents by expanding them into their full forms. This process is crucial for natural language processing tasks where consistency and clarity are important. For example, we will expand `NLP`, `AI`, `DB`, `ETA` into their respective full forms.

### 1. Define Custom Dictionary and Ambiguity Blocklist

We'll start by defining a custom dictionary of common abbreviations and an ambiguity blocklist. Abbreviations in the blocklist will *not* be expanded unless they are explicitly defined within the document itself (e.g., `United States (US)`).

In [8]:
import re

# Custom dictionary for common abbreviations
CUSTOM_ABBREVIATIONS = {
    'AI': 'artificial intelligence',
    'NLP': 'natural language processing',
    'DB': 'database',
    'ETA': 'estimated time of arrival',
    'API': 'application programming interface',
    'CPU': 'central processing unit',
    'RAM': 'random access memory',
    'HTML': 'hypertext markup language',
    'AR' : "approval request",
    'DD' : "Direct Deposit"
}

# Ambiguity blocklist: these will not be expanded by default
AMBIGUITY_BLOCKLIST = {
    'US', 'IT', 'PM', 'MS', 'IR'
}

### 2. Implement the `expand_abbreviations` Function

This function will take a text as input and return the normalized text along with the abbreviation map used for expansion. It will:

1.  Detect document-defined abbreviations (e.g., `Natural Language Processing (NLP)` or `NLP (Natural Language Processing)`).
2.  Combine these with the `CUSTOM_ABBREVIATIONS`.
3.  Iterate through the text and replace recognized abbreviations with their full forms, respecting the `AMBIGUITY_BLOCKLIST`.

In [9]:
def expand_abbreviations(text: str) -> tuple[str, dict]:
    """
    Expands abbreviations and acronyms in a given text.

    Args:
        text: The input text.

    Returns:
        A tuple containing:
        - The normalized text with expanded abbreviations.
        - A dictionary of abbreviations found and their expansions.
    """
    # Create a mutable map for current expansion, starting with custom abbreviations
    current_abbreviation_map = CUSTOM_ABBREVIATIONS.copy()

    # Dictionary to store detected abbreviations in the document (acronym -> full form)
    detected_abbreviations = {}
    normalized_text = text # Start with the original text for initial modifications

    # Regex to find patterns like 'Full Form (ACRONYM)' or 'ACRONYM (Full Form)'
    # Pattern 1: Full Form (ACRONYM)
    pattern1 = re.compile(r'\b([A-Z][a-zA-Z\s]+?)\s*\(([A-Z]{2,})\)')
    # Pattern 2: ACRONYM (Full Form)
    pattern2 = re.compile(r'\b([A-Z]{2,})\s*\(([A-Z][a-zA-Z\s]+?)\)')

    # --- Step 1: Detect and process document-defined abbreviations directly ---
    # Use re.sub with a callback to replace matched patterns and collect abbreviations
    # Document-defined abbreviations should always be expanded to their full form.

    def replace_and_collect_p1(match):
        full_form, acronym = match.groups()
        # If an acronym is explicitly defined in the text, it should always be added to the map
        # and its phrase replaced, regardless of whether it's in AMBIGUITY_BLOCKLIST.
        detected_abbreviations[acronym] = full_form.strip()
        return full_form.strip() # Replace "Full Form (ACRONYM)" with "Full Form"

    normalized_text = pattern1.sub(replace_and_collect_p1, normalized_text)

    def replace_and_collect_p2(match):
        acronym, full_form = match.groups()
        detected_abbreviations[acronym] = full_form.strip()
        return full_form.strip() # Replace "ACRONYM (Full Form)" with "Full Form"

    normalized_text = pattern2.sub(replace_and_collect_p2, normalized_text)

    # --- Step 2: Merge detected abbreviations into the main map ---
    # Document-defined abbreviations take precedence over custom ones.
    current_abbreviation_map.update(detected_abbreviations)

    # --- Step 3: Expand standalone abbreviations using the updated map ---
    # Sort abbreviations by length in descending order to avoid issues with substrings (e.g., 'API' before 'NLP API')
    sorted_abbrs = sorted(current_abbreviation_map.keys(), key=len, reverse=True)

    for abbr in sorted_abbrs:
        full_form = current_abbreviation_map[abbr]

        # Only expand if not in ambiguity blocklist OR if it was explicitly defined in the document
        # (i.e., it was found via pattern1 or pattern2 and added to detected_abbreviations).
        # The first pass already handled 'Full Form (ACRONYM)' structures, so this targets standalone acronyms.
        if abbr not in AMBIGUITY_BLOCKLIST or abbr in detected_abbreviations:
            # Use word boundaries to replace only whole words
            normalized_text = re.sub(r'\b' + re.escape(abbr) + r'\b', full_form, normalized_text)

    # Return the normalized text and the map that was actually used for expansion
    # This map includes CUSTOM_ABBREVIATIONS + document-defined ones.
    return normalized_text, current_abbreviation_map

### 3. Example Usage

Let's test the `expand_abbreviations` function with several example texts.

In [7]:
# Example 1: Basic expansion with custom dictionary
text1 = "The field of AI and NLP is growing rapidly. We need to check the DB for the ETA."
normalized_text1, abbr_map1 = expand_abbreviations(text1)
print("Original Text 1:", text1)
print("Detected Abbreviations 1:", abbr_map1)
print("Normalized Text 1:", normalized_text1)
print("---\n")

# Example 2: Document-defined abbreviation (Full Form (ACRONYM))
text2 = "We are working on Natural Language Processing (NLP) techniques. The API is ready."
normalized_text2, abbr_map2 = expand_abbreviations(text2)
print("Original Text 2:", text2)
print("Detected Abbreviations 2:", abbr_map2)
print("Normalized Text 2:", normalized_text2)
print("---\n")

# Example 3: Document-defined abbreviation (ACRONYM (Full Form))
text3 = "The new feature uses CPU (Central Processing Unit) for better performance. What about RAM?"
normalized_text3, abbr_map3 = expand_abbreviations(text3)
print("Original Text 3:", text3)
print("Detected Abbreviations 3:", abbr_map3)
print("Normalized Text 3:", normalized_text3)
print("---\n")

# Example 4: Ambiguous abbreviation not expanded, and one that is defined
text4 = "The US government is interested in IT. My PM will give an ETA. Also, we use Data Driven (DD) approaches."
normalized_text4, abbr_map4 = expand_abbreviations(text4)
print("Original Text 4:", text4)
print("Detected Abbreviations 4:", abbr_map4)
print("Normalized Text 4:", normalized_text4)
print("---\n")

# Example 5: Multiple abbreviations and complex sentence structure
text5 = "Our team is building an AI system that leverages NLP for text analysis. We store data in a NoSQL DB. The API needs to be robust, and the ETA for completion is tight. I also heard about HTML (HyperText Markup Language) updates."
normalized_text5, abbr_map5 = expand_abbreviations(text5)
print("Original Text 5:", text5)
print("Detected Abbreviations 5:", abbr_map5)
print("Normalized Text 5:", normalized_text5)
print("---\n")

Original Text 1: The field of AI and NLP is growing rapidly. We need to check the DB for the ETA.
Detected Abbreviations 1: {'AI': 'artificial intelligence', 'NLP': 'natural language processing', 'DB': 'database', 'ETA': 'estimated time of arrival', 'API': 'application programming interface', 'CPU': 'central processing unit', 'RAM': 'random access memory', 'HTML': 'hypertext markup language'}
Normalized Text 1: The field of artificial intelligence and natural language processing is growing rapidly. We need to check the database for the estimated time of arrival.
---

Original Text 2: We are working on Natural Language Processing (NLP) techniques. The API is ready.
Detected Abbreviations 2: {'AI': 'artificial intelligence', 'NLP': 'We are working on Natural Language Processing', 'DB': 'database', 'ETA': 'estimated time of arrival', 'API': 'application programming interface', 'CPU': 'central processing unit', 'RAM': 'random access memory', 'HTML': 'hypertext markup language'}
Normalized 

In [10]:
text5 = "After successful validation, submit the changes for approval if required; this step will involve submitting an Approval Request (AR) to the Workflow Queue."
normalized_text5, abbr_map5 = expand_abbreviations(text5)
print("Original Text 5:", text5)
print("Detected Abbreviations 5:", abbr_map5)
print("Normalized Text 5:", normalized_text5)
print("---\n")

Original Text 5: After successful validation, submit the changes for approval if required; this step will involve submitting an Approval Request (AR) to the Workflow Queue.
Detected Abbreviations 5: {'AI': 'artificial intelligence', 'NLP': 'natural language processing', 'DB': 'database', 'ETA': 'estimated time of arrival', 'API': 'application programming interface', 'CPU': 'central processing unit', 'RAM': 'random access memory', 'HTML': 'hypertext markup language', 'AR': 'Approval Request', 'DD': 'Direct Deposit'}
Normalized Text 5: After successful validation, submit the changes for approval if required; this step will involve submitting an Approval Request to the Workflow Queue.
---

